In [ ]:
# Genbank File Handling

In [ ]:
# GenBank is a rich sequence format that stores nucleotide sequences together with extensive biological annotations, including genes, coding regions, references and regulatory elements.

In [ ]:
# Objective:

# This notebook demonstrates how to read, parse and analyse GenBank files using Biopython. It also introduces the NCBI Entrez interface for searching and downloading biological sequence records directly from the NCBI database.

In [ ]:
# Topics covered:

# 1. Introduction to GenBank
# 2. Reading GenBank Files
# 3. GenBank Annotations
# 4. Sequence Features
# 5. Feature Qualifiers
# 6. Extracting Genes
# 7. Introduction to NCBI Entrez
# 8. Searching NCBI
# 9. Downloading GenBank Records
# 10. CDS Extraction
# 11. Protein Translation
# 12. Regulatory Features
# 13. Exporting FASTA

In [ ]:
! pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.3 MB/s eta 0:00:00


In [ ]:
# PART I - Working with Local GenBank File

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving practice.gb to practice.gb


In [ ]:
# Exploring sequence features

In [ ]:
record = SeqIO.read("practice.gb", "genbank")
print(record.id)
print(record.description)
print(record.seq)
print(len(record.seq))

ABC123456.1
Practice gene for learning Biopython
ATGGCTAAATTTGGTTCTCTGCCGGTTATGTGA
33


/usr/local/lib/python3.12/dist-packages/Bio/GenBank/Scanner.py:1537: BiopythonParserWarning: Attempting to parse malformed locus line:
'LOCUS       Gene001                 39 bp    DNA     linear   UNA 01-JAN-2025\n'
Found locus 'Gene001' size '39' residue_type 'DNA'
Some fields may be wrong.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/GenBank/__init__.py:840: BiopythonParserWarning: Expected sequence length 39, found 33 (ABC123456.1).
  warnings.warn(


In [ ]:
# Accessing annotations

In [ ]:
print(record.annotations.keys())

dict_keys(['topology', 'data_file_division', 'date', 'accessions', 'sequence_version', 'keywords', 'source', 'organism', 'taxonomy', 'references', 'molecule_type'])


In [ ]:
print(record.annotations["organism"])
print(record.annotations["molecule_type"])
print(record.annotations["source"])
print(record.annotations["references"])

Synthetic construct Artificial sequences.
DNA
Synthetic construct
[Reference(title='Practice GenBank record', ...)]


In [ ]:
print(len(record.features))

3


In [ ]:
for feature in record.features:
  print(feature.type)

source
gene
CDS


In [ ]:
for feature in record.features:
  print(feature.location)

[0:39](+)
[0:39](+)
[0:39](+)


In [ ]:
for feature in record.features:
  print(feature.type, feature.location)

source [0:39](+)
gene [0:39](+)
CDS [0:39](+)


In [ ]:
# Feature Qualifiers

# Qualifiers provide detailed information about a feature, such as gene names, products, protein identifiers and functional annotations.

In [ ]:
for feature in record.features:          # features are considered as folders and the qualifiers are considered files stored inside these folders
  print(feature.type, feature.location, feature.qualifiers)        # qualifiers are stored as dictionary


source [0:39](+) {'organism': ['Synthetic construct'], 'mol_type': ['genomic DNA']}
gene [0:39](+) {'gene': ['PracticeGene']}
CDS [0:39](+) {'gene': ['PracticeGene'], 'product': ['Practice protein'], 'translation': ['MAKFGSLPVM']}


In [ ]:
for feature in record.features:
  print(feature.type)
  print(feature.qualifiers)
  print("-" * 40)

source
{'organism': ['Synthetic construct'], 'mol_type': ['genomic DNA']}
----------------------------------------
gene
{'gene': ['PracticeGene']}
----------------------------------------
CDS
{'gene': ['PracticeGene'], 'product': ['Practice protein'], 'translation': ['MAKFGSLPVM']}
----------------------------------------


In [ ]:
# Coding Sequences (CDS)

# A coding sequence represents the region of DNA that is translated into a protein.

# Biopython can directly extract and translate CDS regions from annotated GenBank records.

In [ ]:
for feature in record.features:
  if feature.type == "gene":
    print("Gene:", feature.qualifiers["gene"])

  if feature.type == "CDS":
    print("Protein:", feature.qualifiers["product"])
    print("Translation:", feature.qualifiers["translation"])
    print("Translation:", feature.qualifiers["translation"][0])  ## pay attention here

Gene: ['PracticeGene']
Protein: ['Practice protein']
Translation: ['MAKFGSLPVM']
Translation: MAKFGSLPVM


In [ ]:
for feature in record.features:
  if feature.type == "gene":
    print("Gene:", feature.qualifiers["gene"][0])
  if feature.type == "CDS":
    cds = feature.extract(record.seq)
    print(cds)
    print(len(cds))
    protein = cds.translate(to_stop = True)
    print(protein)

Gene: PracticeGene
ATGGCTAAATTTGGTTCTCTGCCGGTTATGTGA
33
MAKFGSLPVM


In [ ]:
# PART II - Retrieving GenBank Records using NCBI Entrez

In [ ]:
from Bio import Entrez
from Bio import SeqIO

Entrez.email = "zaira.hasan656@gmail.com"

In [ ]:
handle = Entrez.esearch(db = "gene", term = "TP53 AND Homo sapiens")
result = Entrez.read(handle)
handle.close()
print(result["Count"])
print(result["IdList"])

2894
['7157', '1956', '348', '7124', '3569', '7422', '7040', '4524', '3091', '2064', '2099', '3586', '6774', '672', '3845', '673', '7421', '29126', '4318', '367']


In [ ]:
genbank_id = "NM_000546"

with Entrez.efetch(db="nucleotide", id=genbank_id, rettype = "gb", retmode = "text") as handle:
  record = SeqIO.read(handle, "genbank")

  handle.close()

In [ ]:
print(record.id)

NM_000546.6


In [ ]:
print(record.description)

Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA


In [ ]:
print(len(record.seq))

2512


In [ ]:
print("First 30 bases in the sequence:", record.seq[:30])

First 30 bases in the sequence: CTCAAAAGTCTAGAGCCACCGTCCAGGGAG


In [ ]:
print(record.annotations.keys())

dict_keys(['molecule_type', 'topology', 'data_file_division', 'date', 'accessions', 'sequence_version', 'keywords', 'source', 'organism', 'taxonomy', 'references', 'comment', 'structured_comment'])


In [ ]:
print(record.annotations["molecule_type"])
print(record.annotations["topology"])
print(record.annotations["keywords"])
print(record.annotations["date"])

mRNA
linear
['RefSeq', 'MANE Select']
21-NOV-2025


In [ ]:
print(len(record.features))

70


In [ ]:
for feature in record.features:
  print(feature.type)
  print(feature.location)
  print(feature.qualifiers)
  print("-" * 40)

source
[0:2512](+)
{'organism': ['Homo sapiens'], 'mol_type': ['mRNA'], 'db_xref': ['taxon:9606'], 'chromosome': ['17'], 'map': ['17p13.1']}
----------------------------------------
gene
[0:2512](+)
{'gene': ['TP53'], 'gene_synonym': ['BCC7; BMFS5; LFS1; P53; TRP53'], 'note': ['tumor protein p53'], 'db_xref': ['GeneID:7157', 'HGNC:HGNC:11998', 'MIM:191170']}
----------------------------------------
exon
[0:114](+)
{'gene': ['TP53'], 'gene_synonym': ['BCC7; BMFS5; LFS1; P53; TRP53'], 'inference': ['alignment:Splign:2.1.0']}
----------------------------------------
misc_feature
[34:37](+)
{'gene': ['TP53'], 'gene_synonym': ['BCC7; BMFS5; LFS1; P53; TRP53'], 'note': ['upstream in-frame stop codon']}
----------------------------------------
exon
[114:216](+)
{'gene': ['TP53'], 'gene_synonym': ['BCC7; BMFS5; LFS1; P53; TRP53'], 'inference': ['alignment:Splign:2.1.0']}
----------------------------------------
CDS
[142:1324](+)
{'gene': ['TP53'], 'gene_synonym': ['BCC7; BMFS5; LFS1; P53; TRP5

In [ ]:
for feature in record.features:
  if feature.type == "gene":
    print("Gene:", feature.qualifiers["gene"][0])

Gene: TP53


In [ ]:
for feature in record.features:
  if feature.type == "CDS":
    cds = feature.extract(record.seq)
    print(cds)
    print(len(cds))
    protein = cds.translate(to_stop = True)
    print(protein)
    print(len(protein))


ATGGAGGAGCCGCAGTCAGATCCTAGCGTCGAGCCCCCTCTGAGTCAGGAAACATTTTCAGACCTATGGAAACTACTTCCTGAAAACAACGTTCTGTCCCCCTTGCCGTCCCAAGCAATGGATGATTTGATGCTGTCCCCGGACGATATTGAACAATGGTTCACTGAAGACCCAGGTCCAGATGAAGCTCCCAGAATGCCAGAGGCTGCTCCCCCCGTGGCCCCTGCACCAGCAGCTCCTACACCGGCGGCCCCTGCACCAGCCCCCTCCTGGCCCCTGTCATCTTCTGTCCCTTCCCAGAAAACCTACCAGGGCAGCTACGGTTTCCGTCTGGGCTTCTTGCATTCTGGGACAGCCAAGTCTGTGACTTGCACGTACTCCCCTGCCCTCAACAAGATGTTTTGCCAACTGGCCAAGACCTGCCCTGTGCAGCTGTGGGTTGATTCCACACCCCCGCCCGGCACCCGCGTCCGCGCCATGGCCATCTACAAGCAGTCACAGCACATGACGGAGGTTGTGAGGCGCTGCCCCCACCATGAGCGCTGCTCAGATAGCGATGGTCTGGCCCCTCCTCAGCATCTTATCCGAGTGGAAGGAAATTTGCGTGTGGAGTATTTGGATGACAGAAACACTTTTCGACATAGTGTGGTGGTGCCCTATGAGCCGCCTGAGGTTGGCTCTGACTGTACCACCATCCACTACAACTACATGTGTAACAGTTCCTGCATGGGCGGCATGAACCGGAGGCCCATCCTCACCATCATCACACTGGAAGACTCCAGTGGTAATCTACTGGGACGGAACAGCTTTGAGGTGCGTGTTTGTGCCTGTCCTGGGAGAGACCGGCGCACAGAGGAAGAGAATCTCCGCAAGAAAGGGGAGCCTCACCACGAGCTGCCCCCAGGGAGCACTAAGCGAGCACTGCCCAACAACACCAGCTCCTCTCCCCAGCCAAAGAAGAAACCACTGGATGGAGAATATTTCACCCTTCAGATCCGTG

In [ ]:
for feature in record.features:
  if feature.type == "CDS":
    stored_protein = feature.qualifiers.get("translation", ["Not available"])[0]
    print(stored_protein)

MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHERCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELPPGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPGGSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD


In [ ]:
for feature in record.features:
  if feature.type == "polyA_site":
    print("Poly A site:", feature.qualifiers["note"][0])

Poly A site: major polyA site


In [ ]:
for feature in record.features:
  if feature.type == "regulatory":
    print("Regulatory class:", feature.qualifiers["regulatory_class"][0])
    print("Shape:", feature.qualifiers["note"][0])

Regulatory class: polyA_signal_sequence
Shape: hexamer: AATAAA


In [ ]:
# FASTA export

In [ ]:
print(record.format("fasta"))

>NM_000546.6 Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA
CTCAAAAGTCTAGAGCCACCGTCCAGGGAGCAGGTAGCTGCTGGGCTCCGGGGACACTTT
GCGTTCGGGCTGGGAGCGTGCTTTCCACGACGGTGACACGCTTCCCTGGATTGGCAGCCA
GACTGCCTTCCGGGTCACTGCCATGGAGGAGCCGCAGTCAGATCCTAGCGTCGAGCCCCC
TCTGAGTCAGGAAACATTTTCAGACCTATGGAAACTACTTCCTGAAAACAACGTTCTGTC
CCCCTTGCCGTCCCAAGCAATGGATGATTTGATGCTGTCCCCGGACGATATTGAACAATG
GTTCACTGAAGACCCAGGTCCAGATGAAGCTCCCAGAATGCCAGAGGCTGCTCCCCCCGT
GGCCCCTGCACCAGCAGCTCCTACACCGGCGGCCCCTGCACCAGCCCCCTCCTGGCCCCT
GTCATCTTCTGTCCCTTCCCAGAAAACCTACCAGGGCAGCTACGGTTTCCGTCTGGGCTT
CTTGCATTCTGGGACAGCCAAGTCTGTGACTTGCACGTACTCCCCTGCCCTCAACAAGAT
GTTTTGCCAACTGGCCAAGACCTGCCCTGTGCAGCTGTGGGTTGATTCCACACCCCCGCC
CGGCACCCGCGTCCGCGCCATGGCCATCTACAAGCAGTCACAGCACATGACGGAGGTTGT
GAGGCGCTGCCCCCACCATGAGCGCTGCTCAGATAGCGATGGTCTGGCCCCTCCTCAGCA
TCTTATCCGAGTGGAAGGAAATTTGCGTGTGGAGTATTTGGATGACAGAAACACTTTTCG
ACATAGTGTGGTGGTGCCCTATGAGCCGCCTGAGGTTGGCTCTGACTGTACCACCATCCA
CTACAACTACATGTGTAACAGTTCCTGCATGGGCGGCATGAACCGGAGGCCCATCCTCAC
CATCAT

In [ ]:
# Summary

# In this notebook, I explored GenBank file analysis and sequence retrieval using Biopython.

# After completing this notebooks, I can:
# - Read GenBank files
# - Access GenBank annotations
# - Explore sequence features
# - Extract gene information
# - Retrieve nucleotide records from NCBI
# - Parse downloaded GenBank files
# - Extract CDS regions
# - Translate coding sequences
# - Export sequences in FASTA format